# Loan Analysis - Model Building

**Models:**
1. Artificial Neural Network (ANN)
2. XGBoost Classifier
3. Random Forest Classifier

We compare all three using ROC-AUC scores.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
)
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC

In [ ]:
# Load processed data
processed = np.load("../data/processed/02_processed_data.npz")
X_train = processed['X_train']
X_test = processed['X_test']
y_train = processed['y_train']
y_test = processed['y_test']

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

In [ ]:
def print_score(true, pred, train=True):
    clf_report = pd.DataFrame(classification_report(true, pred, output_dict=True))
    label = "Train" if train else "Test"
    print(f"{label} Result:")
    print("=" * 50)
    print(f"Accuracy Score: {accuracy_score(true, pred) * 100:.2f}%")
    print("_" * 50)
    print(f"CLASSIFICATION REPORT:\n{clf_report}")
    print("_" * 50)
    print(f"Confusion Matrix:\n{confusion_matrix(true, pred)}\n")

## Artificial Neural Networks (ANNs)

In [ ]:
def plot_learning_evolution(r):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(r.history['loss'], label='Loss')
    axes[0].plot(r.history['val_loss'], label='val_Loss')
    axes[0].set_title('Loss Evolution During Training')
    axes[0].legend()

    axes[1].plot(r.history['AUC'], label='AUC')
    axes[1].plot(r.history['val_AUC'], label='val_AUC')
    axes[1].set_title('AUC Score Evolution During Training')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

def nn_model(num_columns, num_labels, hidden_units, dropout_rates, learning_rate):
    inp = Input(shape=(num_columns,))
    x = BatchNormalization()(inp)
    x = Dropout(dropout_rates[0])(x)
    for i in range(len(hidden_units)):
        x = Dense(hidden_units[i], activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(dropout_rates[i + 1])(x)
    x = Dense(num_labels, activation='sigmoid')(x)

    model = Model(inputs=inp, outputs=x)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=[AUC(name='AUC')])
    return model

In [ ]:
num_columns = X_train.shape[1]
num_labels = 1
hidden_units = [150, 150, 150]
dropout_rates = [0.1, 0, 0.1, 0]
learning_rate = 1e-3

ann_model = nn_model(
    num_columns=num_columns,
    num_labels=num_labels,
    hidden_units=hidden_units,
    dropout_rates=dropout_rates,
    learning_rate=learning_rate
)

r = ann_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32
)

In [ ]:
plot_learning_evolution(r)

In [ ]:
y_train_pred = ann_model.predict(X_train)
print_score(y_train, y_train_pred.round(), train=True)

In [ ]:
y_test_pred = ann_model.predict(X_test)
print_score(y_test, y_test_pred.round(), train=False)

In [ ]:
scores_dict = {
    'ANNs': {
        'Train': roc_auc_score(y_train, ann_model.predict(X_train)),
        'Test': roc_auc_score(y_test, ann_model.predict(X_test)),
    },
}
print(f"ANN ROC-AUC - Train: {scores_dict['ANNs']['Train']:.4f}, Test: {scores_dict['ANNs']['Test']:.4f}")

## XGBoost Classifier

In [ ]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)

y_train_pred = xgb_clf.predict(X_train)
y_test_pred = xgb_clf.predict(X_test)

print_score(y_train, y_train_pred, train=True)
print_score(y_test, y_test_pred, train=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_estimator(
    xgb_clf, X_test, y_test,
    cmap='Blues', values_format='d',
    display_labels=['Default', 'Fully-Paid'], ax=axes[0]
)
axes[0].set_title("XGBoost - Confusion Matrix")

RocCurveDisplay.from_estimator(xgb_clf, X_test, y_test, ax=axes[1])
axes[1].set_title("XGBoost - ROC Curve")

plt.tight_layout()
plt.show()

In [ ]:
scores_dict['XGBoost'] = {
    'Train': roc_auc_score(y_train, xgb_clf.predict(X_train)),
    'Test': roc_auc_score(y_test, xgb_clf.predict(X_test)),
}
print(f"XGBoost ROC-AUC - Train: {scores_dict['XGBoost']['Train']:.4f}, Test: {scores_dict['XGBoost']['Test']:.4f}")

## Random Forest Classifier

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100)
rf_clf.fit(X_train, y_train)

y_train_pred = rf_clf.predict(X_train)
y_test_pred = rf_clf.predict(X_test)

print_score(y_train, y_train_pred, train=True)
print_score(y_test, y_test_pred, train=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_estimator(
    rf_clf, X_test, y_test,
    cmap='Blues', values_format='d',
    display_labels=['Default', 'Fully-Paid'], ax=axes[0]
)
axes[0].set_title("Random Forest - Confusion Matrix")

RocCurveDisplay.from_estimator(xgb_clf, X_test, y_test, ax=axes[1], name='XGBoost')
RocCurveDisplay.from_estimator(rf_clf, X_test, y_test, ax=axes[1], name='Random Forest')
axes[1].set_title("ROC Curves Comparison")

plt.tight_layout()
plt.show()

In [ ]:
scores_dict['Random Forest'] = {
    'Train': roc_auc_score(y_train, rf_clf.predict(X_train)),
    'Test': roc_auc_score(y_test, rf_clf.predict(X_test)),
}
print(f"RF ROC-AUC - Train: {scores_dict['Random Forest']['Train']:.4f}, Test: {scores_dict['Random Forest']['Test']:.4f}")

## Comparing Model Performance

In [ ]:
ml_models = {
    'Random Forest': rf_clf,
    'XGBoost': xgb_clf,
    'ANNs': ann_model
}

for name in ml_models:
    pred = ml_models[name].predict(X_test)
    score = roc_auc_score(y_test, pred)
    print(f"{name:30s} roc_auc_score: {score:.3f}")

In [ ]:
scores_df = pd.DataFrame(scores_dict)

fig, ax = plt.subplots(figsize=(10, 6))
scores_df.T.plot.barh(ax=ax, alpha=0.7)
ax.set_title("ROC-AUC Scores of ML Models")
ax.set_xlabel("ROC-AUC Score")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()